In [1]:
import sys, os, json
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import optuna
import shutil
import math

import pickle
import plotly.io as pio
import matplotlib.pyplot as plt

from pathlib import Path
from dataclasses import asdict

from config import (
    ExperimentConfig,
    TrainingConfig,
    OptunaConfig,
    DDPMTransformerConfig,
    FMConfig,
    AdamConfig,
    AdamWConfig,
    CosineSchedulerConfig,
    WarmupCosineSchedulerConfig,
    PlateauSchedulerConfig,
    CriterionConfig,
)
from engine import Engine
from models import Diffusion, DiffusionTransformer, FlowMatching, DDIM
from utils import (
    setup_logging,
    setup_random_seed,
    ExperimentManager,
    make_all_dataloaders,
    load_variant,
    WindowDataset, make_dataloaders,

    save_data, load_data, build_datasets, load_and_make_dataloaders, verify_roundtrip,
)
from utils.paths import DATASETS_DIR, PROCESSED_DIR, CHECKPOINTS_DIR, EXPERIMENTS_DIR

optuna.logging.set_verbosity(optuna.logging.WARNING)
logger = setup_logging()
logger.info("✓ Imports OK")

2026-06-25 02:03:13,100 - utils.setup - INFO - Logger is set up.
2026-06-25 02:03:13,101 - utils.setup - INFO - ✓ Imports OK


In [2]:
cfg = ExperimentConfig(
    name        = "v1_trial01",
    group_name = "set_index",
    subgroup_name = "set_idx50",
    description = "baseline 33 assets — DDPM",
    random_seed = 72,
    device      = "cuda:0",

    skip_optuna   = True,
    skip_training = False,

    procesed_name = "trial01_w40_assets33_scaler-x_standard_scaler-condrobust",
    processed_group_name = "set_index",
    processed_subgroup_name = "set_idx50",

    model = DDPMTransformerConfig(
        d_model            = 256,
        ff_mult            = 2,
        num_layers         = 3,
        num_attention_heads = 16,
        dropout            = 0.2,
        timesteps          = 1000,
    ),
    # model = FMConfig(
    #     d_model             = 256,
    #     ff_mult             = 2,
    #     num_layers          = 11,
    #     num_attention_heads = 4,
    #     dropout             = 0.1,
    #     sigma_min           = 1e-4,
    #     num_steps           = 100,
    #     solver              = "euler",
    # ),


    training = TrainingConfig(
        num_epochs         = 300,
        # num_epochs         = 1,
        max_grad_norm      = 1.0,
        save_checkpoint_freq = 10,
        # optimizer  = AdamWConfig(lr=1e-4, weight_decay=1e-2),
        # optimizer  = AdamWConfig(lr=1e-4, weight_decay=3.12e-07),
        optimizer  = AdamConfig(lr=0.0004977078401885269, weight_decay=2.2163344843481255e-07),
        scheduler  = WarmupCosineSchedulerConfig(warmup_epochs=20, eta_min=1e-6),
        # scheduler   = PlateauSchedulerConfig( patience = 10, factor = 0.5, eta_min = 1e-6 ),
        criterion  = CriterionConfig("MSELoss"),
    ),
    optuna = OptunaConfig(
        n_trials         = 100,
        epochs_per_trial = 30,
        # n_trials = 1,
        # epochs_per_trial = 1,
        min_resource     = 5,
        max_resource     = 30,
        reduction_factor = 3,
        suggest_d_model  = [128, 256, 512, 1024],
        suggest_ff_mult = [2, 4],
        suggest_n_heads = [4, 8, 16],
        suggest_n_layers = [2, 4, 8, 12],
        suggest_dropout = [0.1, 0.2, 0.3],
        suggest_lr = [1e-4, 5e-4, 1e-3],
        suggest_weight_decay = [1e-7, 1e-6, 1e-5, 1e-4]
    ),
)

In [3]:
display_paths = {
    "cfg.processed_dir": str(cfg.processed_dir),
    "cfg.exp_dir": str(cfg.exp_dir),
    "cfg.checkpoint_dir": str(cfg.checkpoint_dir),
    "cfg.figure_dir": str(cfg.figure_dir),
    "cfg.optuna_dir": str(cfg.optuna_dir),
}
print(json.dumps(display_paths, indent=2, default=str))

{
  "cfg.processed_dir": "/home/narodom.y@FUSION.LAB/research/01_processed/set_index/set_idx50/trial01_w40_assets33_scaler-x_standard_scaler-condrobust",
  "cfg.exp_dir": "/home/narodom.y@FUSION.LAB/research/02_experiments/set_index/set_idx50/v1_trial01_ddpm_warmup-cosine_",
  "cfg.checkpoint_dir": "/home/narodom.y@FUSION.LAB/research/02_experiments/set_index/set_idx50/v1_trial01_ddpm_warmup-cosine_/checkpoints",
  "cfg.figure_dir": "/home/narodom.y@FUSION.LAB/research/02_experiments/set_index/set_idx50/v1_trial01_ddpm_warmup-cosine_/figures",
  "cfg.optuna_dir": "/home/narodom.y@FUSION.LAB/research/02_experiments/set_index/set_idx50/v1_trial01_ddpm_warmup-cosine_/optuna"
}


In [4]:
if not os.path.exists(cfg.exp_dir):
    os.makedirs(cfg.exp_dir)

In [5]:
with open(cfg.exp_dir / "experiment_config.json", "w") as f:
    json.dump(
        asdict(cfg),
        f,
        indent=2,
        ensure_ascii=False,
    )

shutil.copy(
    cfg.processed_dir / "meta.json",
    cfg.exp_dir / "meta.json",
)

print(f"  ✓ experiment_config.json saved")
print(f"  ✓ meta.json copied")

  ✓ experiment_config.json saved
  ✓ meta.json copied


# 2. Load Feature Store

In [6]:
loaded = load_data(str(cfg.processed_dir))

tickers      = loaded["tickers"]
features_lr  = loaded["features_lr"]
features_cond = loaded["features_cond"]
scalers_x    = loaded["scalers_x"]
scalers_cond = loaded["scalers_cond"]

print(f"tickers      : {tickers}")
print(f"features_lr  : {features_lr}")
print(f"features_cond: {features_cond[:5]} ... ({len(features_cond)} total)")

tickers      : ['ADVANC.BK', 'AOT.BK', 'BANPU.BK', 'BBL.BK', 'BDMS.BK', 'BEM.BK', 'BH.BK', 'BJC.BK', 'BTS.BK', 'CENTEL.BK', 'CPALL.BK', 'CPF.BK', 'CPN.BK', 'DELTA.BK', 'EGCO.BK', 'GLOBAL.BK', 'HMPRO.BK', 'IRPC.BK', 'KBANK.BK', 'KKP.BK', 'KTB.BK', 'KTC.BK', 'LH.BK', 'MINT.BK', 'PTT.BK', 'PTTEP.BK', 'RATCH.BK', 'SCC.BK', 'TCAP.BK', 'TISCO.BK', 'TOP.BK', 'TRUE.BK', 'TU.BK']
features_lr  : ['Close', 'High', 'Low', 'Open']
features_cond: ['ADOSC_3_10_minmax', 'ADX_14_minmax', 'ATR_14_minmax', 'BBANDS_lowerband_distance_close', 'BBANDS_middleband_distance_close'] ... (21 total)


In [7]:
cfg.name = f"{cfg.name}_w{loaded['meta']['config']['window_size']}a{len(loaded['meta']['config']['symbols'])}"
cfg.name

'v1_trial01_w40a33'

## 2.1 Shape summary

In [8]:
# x   shape : (N, W, A, C_x)
# cond shape : (N, W, A, C_c)
splits_list = ["train", "val", "test"]
print(f"\n{'split':<8} {'x (N,W,A,C)':<30} {'cond (N,W,A,C_c)':<30} n_windows")
print("-" * 80)
for s in splits_list:
    x    = loaded["splits"][s]["lr"]
    cond = loaded["splits"][s]["cond"]
    print(f"{s:<8} {str(x.shape):<30} {str(cond.shape):<30} {x.shape[0]}")


split    x (N,W,A,C)                    cond (N,W,A,C_c)               n_windows
--------------------------------------------------------------------------------
train    (1852, 40, 33, 4)              (1852, 40, 33, 21)             1852
val      (169, 40, 33, 4)               (169, 40, 33, 21)              169
test     (755, 40, 33, 4)               (755, 40, 33, 21)              755


## 2.2 Scaled check

In [9]:
# x shape: (N, W, A, C)  → ดึง window แรก, timestep แรก → (A, C)
x_train = loaded["splits"]["train"]["lr"]      # (N, W, A, C)
x_sample = x_train[0, 0, :, :]                # (A, C)

print(f"x_train shape: {x_train.shape}")
print(f"x_sample shape (1 timestep, all assets): {x_sample.shape}")

# สถิติ per channel (mean across assets)
print(f"\n{'channel':<20} {'mean':>10} {'std':>10} {'min':>10} {'max':>10}")
print("-" * 60)
x_flat = x_train.reshape(-1, x_train.shape[-1])   # (N*W*A, C)
for i, ch in enumerate(features_lr):
    col = x_flat[:, i]
    print(f"{ch:<20} {col.mean():>10.4f} {col.std():>10.4f} {col.min():>10.4f} {col.max():>10.4f}")

x_train shape: (1852, 40, 33, 4)
x_sample shape (1 timestep, all assets): (33, 4)

channel                    mean        std        min        max
------------------------------------------------------------
Close                   -0.0001     1.0012   -12.8529    10.9220
High                     0.0001     1.0014    -8.8747    11.3928
Low                      0.0001     1.0029   -13.8555    12.4013
Open                     0.0002     1.0013    -9.2404    15.1328


## 2.3 Inverse scale → log returns

In [10]:
# ตรวจ inverse_transform ต่อ ticker แรก
ticker_0 = tickers[0]
ticker_idx = 0

# ดึง window แรก, all timesteps, asset 0 → (W, C)
x_window_scaled = loaded["splits"]["test"]["lr"][0, :, ticker_idx, :]  # (W, C)
scaler = scalers_x[ticker_0]

x_window_inv = scaler.inverse_transform(x_window_scaled)   # (W, C)

print(f"ticker        : {ticker_0}")
print(f"window scaled : {x_window_scaled.shape}   mean={x_window_scaled.mean():.4f}")
print(f"window inv    : {x_window_inv.shape}       mean={x_window_inv.mean():.4f}")

ticker        : ADVANC.BK
window scaled : (40, 4)   mean=-0.1159
window inv    : (40, 4)       mean=-0.0010


## 2.4 Reconstruct close price

In [11]:
# init_price shape: (N, A, C_x)  — price ก่อน window เริ่ม
init_prices = loaded["splits"]["test"]["init_price"]   # (N, A, C_x)
close_feat_idx = features_lr.index("Close")            # หรือ "logdiff_close"

window_idx  = 0
init_price  = init_prices[window_idx, :, close_feat_idx]  # (A,)  — ราคา Close ก่อน window

# log-return close ของทุก asset: (W, A) → transpose → (A, W)
lr_window   = loaded["splits"]["test"]["lr"][window_idx]   # (W, A, C)
lr_close    = lr_window[:, :, close_feat_idx]              # (W, A)

# inverse scale per asset
lr_close_inv = np.stack([
    scalers_x[t].inverse_transform(
        loaded["splits"]["test"]["lr"][window_idx, :, ai, :]
    )[:, close_feat_idx]
    for ai, t in enumerate(tickers)
], axis=1)  # (W, A)

log_price   = np.cumsum(lr_close_inv, axis=0)            # (W, A)
close_recon = init_price[None, :] * np.exp(log_price)    # (W, A)

print(f"\n{'symbol':<15} {'init_price':>12} {'recon_last':>14}")
print("-" * 45)
for ai, sym in enumerate(tickers[:5]):
    print(f"{sym:<15} {init_price[ai]:>12.2f} {close_recon[-1, ai]:>14.2f}")
print("  ... (showing first 5)")


symbol            init_price     recon_last
---------------------------------------------
ADVANC.BK             184.48         178.01
AOT.BK                 61.14          64.76
BANPU.BK                8.02           9.18
BBL.BK                112.39         107.11
BDMS.BK                20.70          23.12
  ... (showing first 5)


## 2.5 DataLoader batch shape check

In [12]:
datasets   = build_datasets(loaded)
dataloaders = make_dataloaders(
    datasets,
    batch_sizes = cfg.training.batch_sizes,
    num_workers = 0,
    pin_memory  = True,
)
print(f"✓ DataLoaders built — train batches: {len(dataloaders['train'])}")

batch = next(iter(dataloaders["train"]))
print(f"\n{'key':<15} {'shape':<35} dtype")
print("-" * 70)
for k, v in batch.items():
    if hasattr(v, "shape"):
        has_nan = torch.isnan(v).any().item() if v.dtype.is_floating_point else False
        has_inf = torch.isinf(v).any().item() if v.dtype.is_floating_point else False
        status  = "✓ clean" if not has_nan and not has_inf else f"✗ nan={has_nan} inf={has_inf}"
        print(f"{k:<15} {str(tuple(v.shape)):<35} {v.dtype}  {status}")
    else:
        print(f"{k:<15} {str(v)[:50]}")

✓ DataLoaders built — train batches: 29

key             shape                               dtype
----------------------------------------------------------------------
x               (64, 40, 33, 4)                     torch.float32  ✓ clean
cond            (64, 40, 33, 21)                    torch.float32  ✓ clean
init_price      (64, 33, 4)                         torch.float32  ✓ clean
date_idx        (64,)                               torch.int64  ✓ clean


# 3. Model Registry

In [13]:
def build_model(exp_cfg, input_dim: int, cond_dim: int) -> nn.Module:
    """
    Build model โดยรับ input_dim และ cond_dim เข้ามาตรง ๆ
    ไม่มี seq_dims/feat_dims อีกแล้ว — caller จัดการ dim เอง

    Parameters
    ----------
    input_dim : int  — C_x  (เช่น จำนวน features ของ x ต่อ 1 token)
    cond_dim  : int  — C_c  (เช่น จำนวน conditioning features ต่อ 1 token)
    """
    m = exp_cfg.model

    if isinstance(m, DDPMTransformerConfig):
        backbone = DiffusionTransformer(
            input_dim           = input_dim,
            cond_dim            = cond_dim,
            d_model             = m.d_model,
            num_layers          = m.num_layers,
            num_attention_heads = m.num_attention_heads,
            dim_feedforward     = m.dim_feedforward,
            dropout             = m.dropout,
        )
        # model = Diffusion(
        #     model      = backbone,
        #     timesteps  = m.timesteps,
        #     beta_start = m.beta_start,
        #     beta_end   = m.beta_end,
        # ).to(exp_cfg.device)
        model = DDIM(
                model      = backbone,
                timesteps  = m.timesteps,
                beta_start = m.beta_start,
                beta_end   = m.beta_end,
            ).to(exp_cfg.device)


    elif isinstance(m, FMConfig):
        backbone = DiffusionTransformer(
            input_dim           = input_dim,
            cond_dim            = cond_dim,
            d_model             = m.d_model,
            num_layers          = m.num_layers,
            num_attention_heads = m.num_attention_heads,
            dim_feedforward     = m.dim_feedforward,
            dropout             = m.dropout,
        )
        model = FlowMatching(
            model     = backbone,
            sigma_min = m.sigma_min,
        ).to(exp_cfg.device)

    else:
        raise ValueError(f"Unknown model config: {type(m)}")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Model     : {m.name}")
    print(f"  input_dim : {input_dim}")
    print(f"  cond_dim  : {cond_dim}")
    print(f"  Params    : {n_params:,}")
    return model

In [14]:
# x shape  : (B, W, A, C_x)  → token = (W*A,), feat = C_x
# cond shape: (B, W, A, C_c)  → token = (W*A,), feat = C_c
# ปรับ input_dim / cond_dim ตาม architecture ที่เลือก
# ตัวอย่างนี้ treat A*W as sequence length, C_x as feature per token
x_shape   = loaded["splits"]["train"]["lr"].shape      # (N, W, A, C_x)
c_shape   = loaded["splits"]["train"]["cond"].shape    # (N, W, A, C_c)

W, A, C_x  = x_shape[1], x_shape[2], x_shape[3]
C_c         = c_shape[3]

input_dim = C_x    # per-token feature dim
cond_dim  = C_c    # per-token conditioning dim

print(f"  W={W}, A={A}, C_x={C_x}, C_c={C_c}")
print(f"  → input_dim={input_dim}, cond_dim={cond_dim}")

model = build_model(cfg, input_dim=input_dim, cond_dim=cond_dim)

  W=40, A=33, C_x=4, C_c=21
  → input_dim=4, cond_dim=21
  Model     : ddpm_transformer
  input_dim : 4
  cond_dim  : 21
  Params    : 4,949,764


# 4. Optuna

## 4.1 Objective function

In [15]:
def objective(trial: optuna.Trial) -> float:
    o = cfg.optuna
    m = cfg.model
    t = cfg.training

    # ── Search space ───────────────────────────────────────────
    d_model  = trial.suggest_categorical("d_model",  o.suggest_d_model)
    ff_mult  = trial.suggest_categorical("ff_mult",  o.suggest_ff_mult)
    n_heads  = trial.suggest_categorical("n_heads",  o.suggest_n_heads)
    n_layers = trial.suggest_int("n_layers", o.suggest_n_layers[0], o.suggest_n_layers[-1])
    dropout  = trial.suggest_float("dropout", o.suggest_dropout[0], o.suggest_dropout[1], step=0.05)
    lr       = trial.suggest_float("lr", o.suggest_lr[0], o.suggest_lr[1], log=True)
    wd       = trial.suggest_float("weight_decay", o.suggest_weight_decay[0], o.suggest_weight_decay[1], log=True)

    if d_model % n_heads != 0:
        raise optuna.exceptions.TrialPruned()

    # ── ตัวแปรที่อาจไม่ถูกสร้างถ้า build_model()/Engine(...) พังก่อนถึงบรรทัดนั้น ──
    # ประกาศไว้ก่อนเป็น None กัน NameError ใน finally
    trial_model = None
    trial_optimizer = None
    trial_scheduler = None
    trial_criterion = None
    engine = None

    try:
        # ── Build trial model ──────────────────────────────────
        # ── Model-specific search space ────────────────────────────
        if isinstance(m, DDPMTransformerConfig):
            trial_model_cfg = DDPMTransformerConfig(
                d_model             = d_model,
                ff_mult             = ff_mult,
                num_layers          = n_layers,
                num_attention_heads = n_heads,
                dropout             = dropout,
                timesteps           = m.timesteps,
                beta_start          = m.beta_start,
                beta_end            = m.beta_end,
            )
        elif isinstance(m, FMConfig):
            sigma_min = trial.suggest_float("sigma_min", o.suggest_sigma_min[0], o.suggest_sigma_min[1], log=True)
            trial_model_cfg = FMConfig(
                d_model             = d_model,
                ff_mult             = ff_mult,
                num_layers          = n_layers,
                num_attention_heads = n_heads,
                dropout             = dropout,
                sigma_min           = sigma_min,
            )
        else:
            raise ValueError(f"Unknown model config: {type(m)}")

        trial_exp_cfg = ExperimentConfig(model=trial_model_cfg, training=t)
        trial_model   = build_model(trial_exp_cfg, input_dim=input_dim, cond_dim=cond_dim)

        trial_optimizer = AdamWConfig(lr=lr, weight_decay=wd).build(trial_model)
        trial_scheduler = t.scheduler.build(trial_optimizer, total_epochs=o.epochs_per_trial)
        trial_criterion = t.criterion.build()

        # ── Mini Engine ─────────────────────────────────────────
        engine = Engine(
            train_loader   = dataloaders["train"],
            val_loader     = dataloaders["val"],
            model          = trial_model,
            optimizer      = trial_optimizer,
            criterion      = trial_criterion,
            scheduler      = trial_scheduler,
            max_grad_norm  = t.max_grad_norm,
            clip_gradients = t.use_clip_grad,
            device         = cfg.device,
            checkpoint_dir = str(cfg.optuna_dir / ".tmp"),
        )

        engine.fit(epochs=o.epochs_per_trial, is_save_best=False, save_every=0, save_plots=False)
        return min(engine.history["val_loss"])

    except torch.cuda.OutOfMemoryError:
        # config สุ่มมาใหญ่เกิน VRAM ที่เหลือไหว ไม่ว่าจะพังตอน build_model(),
        # .to(device), หรือตอน engine.fit() — ดักรวมไว้ที่นี่ทั้งหมด
        # prune trial นี้ทิ้ง ไม่ให้ exception ลามขึ้นไปทำให้ study.optimize() ทั้งก้อนตาย
        raise optuna.exceptions.TrialPruned(
            f"OOM at d_model={d_model}, n_layers={n_layers}, n_heads={n_heads}"
        )

    finally:
        if engine is not None:
            engine.history.clear()
        if trial_model is not None:
            trial_model.cpu()

        del trial_model, trial_optimizer, trial_scheduler, trial_criterion, engine
        plt.close("all")
        gc.collect()
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()  # เคลียร์ peak stat ของ trial นี้ ไม่ให้กระทบ trial ถัดไป


## 4.2 Run study

In [16]:
best_hparams = {}

if not cfg.skip_optuna:
    o = cfg.optuna

    sampler = optuna.samplers.TPESampler(seed=cfg.random_seed)
    pruner  = optuna.pruners.HyperbandPruner(
        min_resource=o.min_resource,
        max_resource=o.max_resource,
        reduction_factor=o.reduction_factor,
    )

    study = optuna.create_study(
        direction  = "minimize",
        study_name = cfg.exp_name,
        sampler    = sampler,
        pruner     = pruner,
    )

    del model
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    study.optimize(objective, n_trials=o.n_trials, show_progress_bar=True)

    # ── Diagnostic: สรุปสถานะ trial ทั้งหมดก่อน — เผื่อ COMPLETE=0 ──
    trials_df = study.trials_dataframe()
    status_counts = trials_df["state"].value_counts()
    print("Trial status summary:")
    print(status_counts.to_string())
    print()

    n_complete = (trials_df["state"] == "COMPLETE").sum()

    if n_complete == 0:
        print("  ⚠ ไม่มี trial ไหน COMPLETE เลย — ดูสาเหตุจาก status summary ด้านบน")
        print("  ดู trials_df ทั้งตารางเพื่อเช็ค fail reason ของแต่ละ trial:")
        print(trials_df[["number", "state", "value"]].to_string())
    else:
        best_hparams = study.best_params

        print(f"\n  Best val loss : {study.best_value:.6f}")
        print(f"  Best params   :\n{json.dumps(best_hparams, indent=4)}")

else:
    print("skip_optuna=True — using exp_cfg params")

skip_optuna=True — using exp_cfg params


## 4.3 Save Optuna results

In [17]:
# if not cfg.skip_optuna:
#     tmp = cfg.optuna_dir / ".tmp"
#     if tmp.exists():
#         shutil.rmtree(tmp)

#     with open(cfg.optuna_dir / "study.pkl", "wb") as f:
#         pickle.dump(study, f)

#     best_params_out = {
#         **best_hparams,
#         "dim_feedforward": best_hparams["d_model"] * best_hparams["ff_mult"],
#         "best_val_loss"  : study.best_value,
#     }
#     with open(cfg.optuna_dir / "best_params.json", "w") as f:
#         json.dump(best_params_out, f, indent=2)

#     trials_df = study.trials_dataframe()
#     trials_df.to_csv(cfg.optuna_dir / "all_trials.csv", index=False)

#     plots = {
#         "optimization_history" : optuna.visualization.plot_optimization_history(study),
#         "parallel_coordinate"  : optuna.visualization.plot_parallel_coordinate(study),
#         "param_importances"    : optuna.visualization.plot_param_importances(study),
#     }
#     for name, fig in plots.items():
#         pio.write_html(fig, str(cfg.optuna_dir / f"{name}.html"))

#     print(f"  ✓ study.pkl")
#     print(f"  ✓ best_params.json — best val loss: {study.best_value:.6f}")
#     print(f"  ✓ all_trials.csv   — {len(trials_df)} trials")
#     print(f"  → {cfg.optuna_dir}")
if not cfg.skip_optuna:
    tmp = cfg.optuna_dir / ".tmp"
    if tmp.exists():
        shutil.rmtree(tmp)

    with open(cfg.optuna_dir / "study.pkl", "wb") as f:
        pickle.dump(study, f)

    trials_df = study.trials_dataframe()
    trials_df.to_csv(cfg.optuna_dir / "all_trials.csv", index=False)

    # ── Guard: ถ้าไม่มี complete trial เลย ไม่ต้อง save best_params ──
    if best_hparams:
        best_params_out = {
            **best_hparams,
            "dim_feedforward": best_hparams["d_model"] * best_hparams["ff_mult"],
            "best_val_loss"  : study.best_value,
        }
        with open(cfg.optuna_dir / "best_params.json", "w") as f:
            json.dump(best_params_out, f, indent=2)
        print(f"  ✓ best_params.json — best val loss: {study.best_value:.6f}")
    else:
        print("  ⚠ best_params.json ไม่ถูก save — ไม่มี trial ที่ COMPLETE")

    plots = {
        "optimization_history" : optuna.visualization.plot_optimization_history(study),
        "parallel_coordinate"  : optuna.visualization.plot_parallel_coordinate(study),
        "param_importances"    : optuna.visualization.plot_param_importances(study),
    }
    for name, fig in plots.items():
        pio.write_html(fig, str(cfg.optuna_dir / f"{name}.html"))

    print(f"  ✓ study.pkl")
    print(f"  ✓ all_trials.csv   — {len(trials_df)} trials")
    print(f"  → {cfg.optuna_dir}")

## 4.4 GPU Cleanup (post-Optuna)

In [18]:
# Optuna trial ก่อนหน้าอาจมี tensor/state ค้างอยู่บน VRAM
# (เช่น optimizer state, autograd graph จาก trial ที่ OOM, หรือ study object เอง)
# เคลียรตรงนี้ก่อนเข้า 5.1 build final model กัน VRAM เต็มตอน .fit()

print("Before cleanup:")
print(f"  Allocated : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"  Reserved  : {torch.cuda.memory_reserved() / 1e9:.2f} GB")

if "study" in dir():
    del study
if "sampler" in dir():
    del sampler
if "pruner" in dir():
    del pruner

gc.collect()
gc.collect()  # เรียกซ้ำกัน object graph ที่มี circular ref ยังไม่โดนรอบแรก
torch.cuda.synchronize()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("\nAfter cleanup:")
print(f"  Allocated : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"  Reserved  : {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Before cleanup:
  Allocated : 0.02 GB
  Reserved  : 0.02 GB

After cleanup:
  Allocated : 0.02 GB
  Reserved  : 0.02 GB


# 5. Fit Model (Training)

## 5.1 Build Final Model

In [19]:
if not cfg.skip_training:

    if best_hparams:
        if isinstance(cfg.model, DDPMTransformerConfig):
            final_model_cfg = DDPMTransformerConfig(
                d_model             = best_hparams["d_model"],
                ff_mult             = best_hparams["ff_mult"],
                num_layers          = best_hparams["n_layers"],
                num_attention_heads = best_hparams["n_heads"],
                dropout             = best_hparams["dropout"],
                timesteps           = cfg.model.timesteps,
                beta_start          = cfg.model.beta_start,
                beta_end            = cfg.model.beta_end,
            )
        elif isinstance(cfg.model, FMConfig):
            final_model_cfg = FMConfig(
                d_model             = best_hparams["d_model"],
                ff_mult             = best_hparams["ff_mult"],
                num_layers          = best_hparams["n_layers"],
                num_attention_heads = best_hparams["n_heads"],
                dropout             = best_hparams["dropout"],
                sigma_min           = best_hparams["sigma_min"],
                num_steps           = cfg.model.num_steps,
                solver              = cfg.model.solver,
            )
        # ✅ เพิ่มตรงนี้
        final_optimizer_cfg = AdamWConfig(
            lr           = best_hparams["lr"],
            weight_decay = best_hparams["weight_decay"],
        )
        print("  ✓ Using Optuna best_hparams")
    else:
        final_model_cfg     = cfg.model
        # ✅ เพิ่มตรงนี้
        final_optimizer_cfg = cfg.training.optimizer
        print("  ✓ Using cfg defaults (skip_optuna=True)")

    final_model = build_model(
        ExperimentConfig(model=final_model_cfg, training=cfg.training),
        input_dim = input_dim,
        cond_dim  = cond_dim,
    )

    print(f"  d_model  : {final_model_cfg.d_model}")
    print(f"  n_layers : {final_model_cfg.num_layers}")
    print(f"  n_heads  : {final_model_cfg.num_attention_heads}")
    print(f"  dropout  : {final_model_cfg.dropout}")
    print(f"  lr       : {final_optimizer_cfg.lr:.2e}")  # ✅ uncomment ได้แล้ว

  ✓ Using cfg defaults (skip_optuna=True)
  Model     : ddpm_transformer
  input_dim : 4
  cond_dim  : 21
  Params    : 4,949,764
  d_model  : 256
  n_layers : 3
  n_heads  : 16
  dropout  : 0.2
  lr       : 4.98e-04


## 5.2 Train

In [20]:
if not cfg.skip_training:
    t = cfg.training

    optimizer = final_optimizer_cfg.build(final_model)
    scheduler = t.scheduler.build(optimizer, total_epochs=t.num_epochs)
    criterion = t.criterion.build()

    engine = Engine(
        train_loader   = dataloaders["train"],
        val_loader     = dataloaders["val"],
        model          = final_model,
        optimizer      = optimizer,
        criterion      = criterion,
        scheduler      = scheduler,
        max_grad_norm  = t.max_grad_norm,
        clip_gradients = t.use_clip_grad,
        device         = cfg.device,
        checkpoint_dir = str(cfg.checkpoint_dir),
    )

    engine.fit(
        epochs       = t.num_epochs,
        is_save_best = True,
        save_every   = t.save_checkpoint_freq,
    )

    print(f"\n  ✓ Training complete")
    print(f"  ✓ Best  → {cfg.checkpoint_dir / 'best_model.pt'}")

2026-06-25 02:03:23,006 - Engine - INFO - Engine initialised  device=cuda:0


2026-06-25 02:03:23,006 - Engine - INFO - Engine initialised  device=cuda:0


2026-06-25 02:03:23,009 - Engine - INFO - Criterion : MSELoss


2026-06-25 02:03:23,009 - Engine - INFO - Criterion : MSELoss


2026-06-25 02:03:23,010 - Engine - INFO - Checkpoints → /home/narodom.y@FUSION.LAB/research/02_experiments/set_index/set_idx50/v1_trial01_w40a33_ddpm_warmup-cosine_/checkpoints


...

2026-06-25 03:13:17,467 - Engine - INFO - Epoch 300 | Val Loss: 0.5545


2026-06-25 03:13:17,946 - Engine - INFO - Training complete.


2026-06-25 03:13:17,946 - Engine - INFO - Training complete.


2026-06-25 03:13:20,562 - Engine - INFO - All plots saved → /home/narodom.y@FUSION.LAB/research/02_experiments/set_index/set_idx50/v1_trial01_w40a33_ddpm_warmup-cosine_/plots


2026-06-25 03:13:20,562 - Engine - INFO - All plots saved → /home/narodom.y@FUSION.LAB/research/02_experiments/set_index/set_idx50/v1_trial01_w40a33_ddpm_warmup-cosine_/plots

  ✓ Training complete
  ✓ Best  → /home/narodom.y@FUSION.LAB/research/02_experiments/set_index/set_idx50/v1_trial01_w40a33_ddpm_warmup-cosine_/checkpoints/best_model.pt


## 5.3 Load best checkpoint & sanity check

In [21]:
# Load best checkpoint
engine.load_checkpoint("best_model.pt")
print("✓ Best checkpoint loaded")

# ── Single test batch sanity check ────────────────────────
batch_test = next(iter(dataloaders["test"]))
x_test    = batch_test["x"].to(cfg.device)      # (B, W, A, C_x)
cond_test = batch_test["cond"].to(cfg.device)   # (B, W, A, C_c)

B, W, A, C_x = x_test.shape
_, _, _, C_c  = cond_test.shape

# ส่ง 4D ตรงๆ — DiffusionTransformer.forward() expect (B, W, A, C)
with torch.no_grad():
    # x_gen = engine.model.sample(
    #     x_cond       = cond_test,
    #     output_shape = x_test.shape,
    # )
    if isinstance(cfg.model, DDPMTransformerConfig):
        x_gen = engine.model.sample(
            x_cond       = cond_test,
            output_shape = x_test.shape,
            ddim_steps   = 50,
            eta          = 0.0,
        )
    elif isinstance(cfg.model, FMConfig):
        x_gen = engine.model.sample(
            x_cond       = cond_test,
            output_shape = x_test.shape,
            num_steps    = cfg.model.num_steps,
            method       = cfg.model.solver,
        )

print(f"  input  shape : {tuple(x_test.shape)}")
print(f"  output shape : {tuple(x_gen.shape)}")
print(f"  output mean  : {x_gen.mean().item():.4f}")
print(f"  output std   : {x_gen.std().item():.4f}")
print(f"  has nan      : {torch.isnan(x_gen).any().item()}")
print(f"  has inf      : {torch.isinf(x_gen).any().item()}")

assert x_gen.shape == x_test.shape, f"Shape mismatch: {x_gen.shape} vs {x_test.shape}"
print("✓ Shape assertion passed")

/home/narodom.y@FUSION.LAB/research/src/engine/trainer.py:285: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=self.device)
2026-06-25 03:

2026-06-25 03:13:20,989 - Engine - INFO - Loaded checkpoint: /home/narodom.y@FUSION.LAB/research/02_experiments/set_index/set_idx50/v1_trial01_w40a33_ddpm_warmup-cosine_/checkpoints/best_model.pt
✓ Best checkpoint loaded
  input  shape : (1, 40, 33, 4)
  output shape : (1, 40, 33, 4)
  output mean  : 0.1605
  output std   : 0.5615
  has nan      : False
  has inf      : False
✓ Shape assertion passed
